In [51]:
import pandas as pd

In [52]:
label_name_path = "csv/class-descriptions-boxable.csv"
label_df = pd.read_csv(
    label_name_path,
    header=None,
    names=["mid", "name"]
)

In [53]:
print("===== LABEL DESCRIPTIONS =====")
print(label_df.columns)
print(label_df)

===== LABEL DESCRIPTIONS =====
Index(['mid', 'name'], dtype='str')
             mid        name
0      /m/011k07    Tortoise
1    /m/011q46kg   Container
2      /m/012074      Magpie
3      /m/0120dh  Sea turtle
4      /m/01226z    Football
..           ...         ...
596     /m/0qmmr  Wheelchair
597   /m/0wdt60w  Rugby ball
598      /m/0xfy   Armadillo
599     /m/0xzly     Maracas
600     /m/0zvk5      Helmet

[601 rows x 2 columns]


In [54]:
target_labels = [
    # Workspace Safety
    "Coffee cup", "Wine glass", "Laptop", "Power plugs and sockets", "Desk",

    # Kitchen Safety
    "Kitchen knife", "Paper towel", "Gas stove", "Table",

    # Toddler/Pet Safety
    "Coin", "Toy", "Houseplant", "Dog", "Boy", "Girl"
]

safety_mids = set(
    label_df[label_df["name"].str.lower().isin([l.lower() for l in target_labels])]
    ["mid"]
)

print("Safety mids:", safety_mids)

Safety mids: {'/m/01c648', '/m/0242l', '/m/02wv84t', '/m/03fp41', '/m/02p5f1q', '/m/03bbps', '/m/01bl7v', '/m/04bcr3', '/m/05r655', '/m/058qzx', '/m/0138tl', '/m/01y9k5', '/m/02w3r3', '/m/0bt9lr', '/m/09tvcd'}


In [55]:
print("===== CHECK LABEL EXISTENCE =====")
for label in target_labels:
    match = label_df[label_df["name"].str.lower() == label.lower()]
    if not match.empty:
        print(f"✅ FOUND: {label} → mid = {match.iloc[0]['mid']}")
    else:
        print(f"❌ NOT FOUND: {label}")

===== CHECK LABEL EXISTENCE =====
✅ FOUND: Coffee cup → mid = /m/02p5f1q
✅ FOUND: Wine glass → mid = /m/09tvcd
✅ FOUND: Laptop → mid = /m/01c648
✅ FOUND: Power plugs and sockets → mid = /m/03bbps
✅ FOUND: Desk → mid = /m/01y9k5
✅ FOUND: Kitchen knife → mid = /m/058qzx
✅ FOUND: Paper towel → mid = /m/02w3r3
✅ FOUND: Gas stove → mid = /m/02wv84t
✅ FOUND: Table → mid = /m/04bcr3
✅ FOUND: Coin → mid = /m/0242l
✅ FOUND: Toy → mid = /m/0138tl
✅ FOUND: Houseplant → mid = /m/03fp41
✅ FOUND: Dog → mid = /m/0bt9lr
✅ FOUND: Boy → mid = /m/01bl7v
✅ FOUND: Girl → mid = /m/05r655


In [56]:
def extract_ids_large(df, prefix, target_mids):
    mask = df["LabelName"].isin(target_mids)
    ids = df.loc[mask, "ImageID"].dropna().drop_duplicates()
    return (prefix + "/" + ids).tolist()

In [57]:
train_path = "csv/oidv6-train-annotations-bbox.csv"
val_path = "csv/validation-annotations-bbox.csv"
test_path = "csv/test-annotations-bbox.csv"
output_path = "csv/image_lst.txt"

In [58]:
cols = ["ImageID", "LabelName", "XMin", "XMax", "YMin", "YMax"]

train_df = pd.read_csv(train_path, usecols=cols)
val_df   = pd.read_csv(val_path, usecols=cols)
test_df  = pd.read_csv(test_path, usecols=cols)

In [59]:
train_ids = extract_ids_large(train_df, "train", safety_mids)
val_ids   = extract_ids_large(val_df, "validation", safety_mids)
test_ids  = extract_ids_large(test_df, "test", safety_mids)

In [60]:
all_ids = train_ids + val_ids + test_ids

with open(output_path, "w") as f:
    for img_path in all_ids:
        f.write(img_path + "\n")

print(f"Saved {len(all_ids)} image paths to {output_path}")

Saved 252855 image paths to ./image_lst.txt


In [61]:
print("===== TRAIN HEAD =====")
print(train_df.head())

print("\n===== VALIDATION HEAD =====")
print(val_df.head())

print("\n===== TEST HEAD =====")
print(test_df.head())

===== TRAIN HEAD =====
            ImageID  LabelName      XMin      XMax      YMin      YMax
0  000002b66c9c498e  /m/01g317  0.012500  0.195312  0.148438  0.587500
1  000002b66c9c498e  /m/01g317  0.025000  0.276563  0.714063  0.948438
2  000002b66c9c498e  /m/01g317  0.151562  0.310937  0.198437  0.590625
3  000002b66c9c498e  /m/01g317  0.256250  0.429688  0.651563  0.925000
4  000002b66c9c498e  /m/01g317  0.257812  0.346875  0.235938  0.385938

===== VALIDATION HEAD =====
            ImageID LabelName      XMin      XMax      YMin      YMax
0  0001eeaf4aed83f9  /m/0cmf2  0.022673  0.964201  0.071038  0.800546
1  000595fe6fee6369  /m/02wbm  0.000000  1.000000  0.000000  1.000000
2  000595fe6fee6369  /m/02xwb  0.141384  0.179676  0.676275  0.731707
3  000595fe6fee6369  /m/02xwb  0.213549  0.253314  0.299335  0.354767
4  000595fe6fee6369  /m/02xwb  0.232695  0.288660  0.490022  0.545455

===== TEST HEAD =====
            ImageID LabelName      XMin      XMax      YMin      YMax
0  000026